# 4 · Electron depth and the exobase

Figures built from the `Electron Depth Dist` variable: how collisional the electrons are as a
function of where they were produced.

* `plot_el_depth_2d` / `grouped_el_depth_2d_plot` – `make_2dElDepthPlot`, `make_grouped_ElDepth2d_Plot`
* `el_depth_angle_maps_plot` – `make_eDepth_QVec_2d_AllCases_Plot`
* `el_depth_summary_plot` – `make_eDepth_QVec_summaryPlot`
* `el_depth_distribution_plot` – `plot_eDepth_dist`

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

import collide_comet as cc
from collide_comet import plotting as ccp
from run_config import get_cases, make_saver

cases, DEMO = get_cases()
save = make_saver("electron_depth", DEMO)
cases

## The raw distribution

`case.electron_depth()` returns the radial bin edges, the depth grid and the distribution with shape
`(n_depth, n_process_groups, n_r, n_theta, 3)`; the last axis is the polar axis (x, y, z) the angle is
measured from. As in the MATLAB loader the depth grid is `logspace(-5, 4, n)`.

In [ ]:
case = cases["-X"]
rad_edges, depth_grid, depth_dist = case.electron_depth("total")     # PE + SW
print("radial edges:", rad_edges.shape, " depth grid:", depth_grid.shape, " distribution:", depth_dist.shape)

## Median depth about one axis

The black curve is the smoothed hull of the region with median depth > 1, i.e. the electron exobase.

In [ ]:
fig, ax, cbar, depth = ccp.plot_el_depth_2d(rad_edges, depth_grid, depth_dist, i_dim="X", clim=(-2, 2))
ax.set_title(f"Case {case.name}", pad=14)
save(fig, f"el_depth_2d_case{case.name}_X");

In [ ]:
for axis in ["X", "Y", "Z"]:
    fig, axes, cbar, depth_all = ccp.grouped_el_depth_2d_plot(cases, i_dim=axis, clim=(-2, 0.5))
    save(fig, f"el_depth_2d_{axis}")

## Depth versus angle to the outgassing axis

Maps of the median depth against radius and angle for each case, plus the median depth in a shell
(left, bottom) compared with the reference case. Dashed: analytic exobase for the local outgassing
rate. Solid: outermost radius where the median depth exceeds `depth_thresh`.

In [ ]:
fig, axes, depth_maps = ccp.el_depth_angle_maps_plot(
    cases, r0=[30], dr=20, clim=(-2, 1), r_max=300, depth_thresh=1,
)
save(fig, "el_depth_angle_maps");

## Electron depth summary

`make_eDepth_QVec_summaryPlot`: the median electron depth in one shell against the angle to maximum outgassing for every case, and the radius–angle map of a single case (`case_2d`).

In [ ]:
fig, axes, depth_summary = ccp.el_depth_summary_plot(
    cases, r0=50, dr=10,
    n_angle_bins=None,   # None = native angle bins; e.g. 18 re-bins to 10-degree bins
    case_2d="Ref",       # which case to show in the map
)
save(fig, "el_depth_summary");

## Distribution of electron depth

Median and interquartile range per case (top) and the full distributions (bottom).

In [ ]:
fig, (ax_top, ax), dist = ccp.el_depth_distribution_plot(cases, i_xs=2, source="pe", max_radius=200)
save(fig, "el_depth_distribution")

for name, d in dist.items():
    print(f"{name:>4s}: median tau = {d['median']:.3g}   IQR = [{d['q1']:.3g}, {d['q3']:.3g}]")

## Exobase estimates as numbers

`collide_comet.analysis.median_depth_and_exobase` is the port of `compute_2dElDepthMedianAndExobase`
and returns the hull and the envelope $r_{exo}(\theta)$ without drawing anything.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.2), constrained_layout=True)
for case in cases:
    axis = int(np.argmax(np.abs(case.outgassing_raw.q_vec))) if not case.is_reference else 0
    out = cc.analysis.median_depth_and_exobase(*case.electron_depth("total"), i_dim=axis)
    env = out["exo"]["envelope"]
    ax.plot(np.degrees(env["theta"]), env["r_smooth"], label=f"{case.name} (about {'XYZ'[axis]})")
ax.set_xlabel("Polar angle from + axis [deg]")
ax.set_ylabel("Exobase distance [km]")
ax.set_xlim(0, 180)
ax.legend(fontsize=9)
save(fig, "exobase_envelopes");